# 03 Feature engineering
This notebook will load the structured parquets from `02_preprocess_data.ipynb` and do the following:
1. Feature engineer `disruption_severity` for the `mta_alerts` dataset, as a measure of extent of disruption to general commuters per (hour, zone)
2. Combine datasets from tlc_data and aggregate for `n_trips` and `gross_fare` per (hour, zone, service_type) 
3. Use taxi zones and `latitude`, `longitude` of distinct `station_complex_id` from `mta_ridership` to determine `n_stations` per zone 
4. Use `affected` from `mta_alerts` and `latitude`, `longitude` from `mta_ridership` to determine `n_alerts` per (hour, zone)
5. Use `coco` from `meteostat_weather`, mapped to a `weather_severity` (categorical: normal -> severe) per hour

In [1]:
# Restart spark if required
try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "America/New_York") # timezone set to NYC
    .getOrCreate()
)
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.adaptive.enabled", "true")

spark.sparkContext.setLogLevel("ERROR") # silence warnings

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/15 03:33:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Set up and configuration

In [ ]:
# Imports
from pathlib import Path
from pyspark.sql import functions as F
import geopandas as gpd
import re

In [ ]:
# Raw spatial data from `01_fetch_data.ipynb`
taxi_zones_path = "../data/raw/taxi_zones/taxi_zones.shp"

# Input directories from `02_preprocess_data.ipynb`
processed_dir = "../data/processed"

yellow_path = f"{processed_dir}/tlc_trips/yellow"
green_path = f"{processed_dir}/tlc_trips/green"
fhvhv_path = f"{processed_dir}/tlc_trips/fhvhv"
ridership_path = f"{processed_dir}/mta_ridership"
alerts_path = f"{processed_dir}/mta_alerts"
weather_path = f"{processed_dir}/meteostat_weather"

# Output directories for `03_feature_engineering.ipynb`
features_dir = "../data/features"

In [5]:
# Helper function: prints shape of sdf
def spark_shape(sdf):
    return (sdf.count(), len(sdf.columns))

# Helper function: save a sdf as parquet
def save_sdf(sdf, name):
    path = f"{features_dir}/{name}"
    sdf.write.mode("overwrite").parquet(path)
    print(f"Saved {name} to {path}")

# Helper function: read a sdf from parquet
def read_sdf(name):
    return spark.read.parquet(f"{features_dir}/{name}")

## Extract TLC Trip Record features
In this section, we will aggregate and extract all the TLC data that we need as features, grouped by (PULocationID, hour)
- `n_trips_taxi`: total number of trips taken by taxi (yellow, green) in the given zone (PULocationID) during the given hour
- `gross_fare_taxi`: total amount paid by passenger(s), excluding any tips, summed over all trips by taxi in the given zone (PULocationID) during the given hour
- `n_trips_fhvhv`: total number of trips taken by FHVHV (Uber, Lyft) in the given zone (PULocationID) during the given hour
- `gross_fare_fhvhv`: total amount paid by passenger(s), excluding any tips, summed over all trips by FHVHV in the given zone (PULocationID) during the given hour
- `avg_wait_time`: average wait time (pickup time - request time) for FHVHV, in the given zone (PULocationID) during the given hour (determined by request time)

In [20]:
# Read processed TLC data
sdf_yellow = spark.read.parquet(yellow_path)
sdf_green = spark.read.parquet(green_path)
sdf_fhvhv = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(fhvhv_path)
)

In [6]:
# Stack yellow + green into one taxi df 
sdf_taxi = sdf_yellow.unionByName(sdf_green) # match columns by union of names
sdf_taxi.show(5)
spark_shape(sdf_taxi)

+-------------------+-------------------+------------+------------+--------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|subtotal|service_type|
+-------------------+-------------------+------------+------------+--------+------------+
|2026-04-01 00:40:05|2026-04-01 00:52:44|         237|          68|   21.35|      yellow|
|2026-04-01 00:09:19|2026-04-01 00:21:29|         138|          75|   45.16|      yellow|
|2026-04-01 00:15:29|2026-04-01 00:34:14|         138|         112|   41.20|      yellow|
|2026-04-01 00:04:53|2026-04-01 00:11:54|         230|         234|   14.35|      yellow|
|2026-04-01 00:37:00|2026-04-01 00:45:17|          68|         107|   16.45|      yellow|
+-------------------+-------------------+------------+------------+--------+------------+
only showing top 5 rows


(47592720, 6)

In [7]:
# Aggregate taxi trips by pickup zone-hour
sdf_taxi_hourly = (
    sdf_taxi
    .withColumn("hour", F.date_trunc("hour", F.col("pickup_datetime")))
    .groupBy("hour", "PULocationID")
    .agg(
        F.count("*").alias("n_trips_taxi"),
        F.round(F.sum("subtotal"), 2).alias("gross_fare_taxi"),
    )
)
sdf_taxi_hourly.show(5)
spark_shape(sdf_taxi_hourly)

+-------------------+------------+------------+---------------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|
+-------------------+------------+------------+---------------+
|2026-04-01 05:00:00|         224|           3|          57.75|
|2026-04-01 14:00:00|          90|          64|        1276.57|
|2026-04-01 14:00:00|          24|          24|         444.76|
|2026-04-02 05:00:00|         151|           3|          60.00|
|2026-04-02 15:00:00|         168|           1|          12.20|
+-------------------+------------+------------+---------------+
only showing top 5 rows


(1041794, 4)

In [8]:
# Aggregate FHVHV trips by pickup zone-hour
sdf_fhvhv_hourly = (
    sdf_fhvhv
    .withColumn("hour", F.date_trunc("hour", F.col("request_datetime"))) # for FHVHV, use request_datetime to match demand time
    .withColumn(
        "wait_time",
        (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")) # wait_time in seconds
    )
    .groupBy("hour", "PULocationID")
    .agg(
        F.count("*").alias("n_trips_fhvhv"),
        F.round(F.sum("subtotal"), 2).alias("gross_fare_fhvhv"),
        F.round(F.avg("wait_time"), 2).alias("avg_wait_time_fhvhv"),
    )
)
sdf_fhvhv_hourly.show(5)
spark_shape(sdf_fhvhv_hourly)

+-------------------+------------+-------------+----------------+-------------------+
|               hour|PULocationID|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|
+-------------------+------------+-------------+----------------+-------------------+
|2025-06-04 18:00:00|          26|          234|         4197.35|              272.0|
|2025-06-05 06:00:00|          40|           38|         1763.93|             243.13|
|2025-06-05 09:00:00|           5|            6|          167.62|              267.0|
|2025-06-05 12:00:00|          32|           85|         1737.86|             267.38|
|2025-06-05 17:00:00|         164|          379|        27380.30|             447.07|
+-------------------+------------+-------------+----------------+-------------------+
only showing top 5 rows


(3123806, 5)

In [9]:
# Combine the two hourly aggregates
sdf_tlc_hourly = (
    sdf_taxi_hourly
    .join(sdf_fhvhv_hourly, on=["hour", "PULocationID"], how="full")
    .fillna({
        "n_trips_taxi": 0,
        "gross_fare_taxi": 0.0,
        "n_trips_fhvhv": 0,
        "gross_fare_fhvhv": 0.0,
    }) # do not fillna for avg_wait_time: yellow/green don't have wait_time
)
#sdf_tlc_hourly.show(5)
spark_shape(sdf_tlc_hourly)

(3138172, 7)

## Create zone-station dataframe
We need a zone-station dataframe to map `mta_ridership` data with respect to `taxi_zones`. This section will produce
- `sdf_station_zones`: zone-station dataframe which maps every distinct `station_complex_id` to a `PULocationID`, also containing interpretable columns such as `station_complex` (station name), `taxi_zone` (zone name), `_borough` (from both datasets to ensure consistency), `latitude` and `longitude`

- `sdf_ridership_station`: `PULocationID` merged with `sdf_ridership` using the mapping provided in `sdf_station_zones` 

In [29]:
# Read MTA ridership
sdf_ridership = spark.read.parquet(ridership_path)

# Read taxi zones
zones = gpd.read_file(taxi_zones_path)[["LocationID", "borough", "zone", "geometry"]]

In [30]:
# Get unique station locations from mta_ridership
stations = sdf_ridership.select(
    "station_complex_id",
    "station_complex",
    F.col("borough").alias("mta_borough"), # avoid column name clash
    "latitude",
    "longitude"
).distinct().toPandas()

spark_shape(stations)

(station_complex_id    424
 station_complex       424
 mta_borough           424
 latitude              424
 longitude             424
 dtype: int64,
 5)

In [32]:
# Turn station latitude/longitude into points
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]),
    crs="EPSG:4326"    # standard GPS-style coordinates
).to_crs(zones.crs)

# Spatial join station with taxi zone with gpd, then convert back to Spark
sdf_station_zones = spark.createDataFrame(
    gpd.sjoin(stations, zones, predicate="within")
    [[
        "station_complex_id",
        "station_complex",
        "mta_borough",
        "latitude",
        "longitude",
        "LocationID",
        "borough",
        "zone",
    ]]
    .rename(columns={
        "LocationID": "PULocationID",
        "borough": "taxi_borough", # specify that it came from taxi dataset
        "zone": "taxi_zone",
    })
)
sdf_station_zones.show(5)
spark_shape(sdf_station_zones)

+------------------+--------------------+-----------+---------+----------+------------+------------+------------------+
|station_complex_id|     station_complex|mta_borough| latitude| longitude|PULocationID|taxi_borough|         taxi_zone|
+------------------+--------------------+-----------+---------+----------+------------+------------+------------------+
|                72|Fort Hamilton Pkw...|   Brooklyn|40.631386| -74.00535|          26|    Brooklyn|      Borough Park|
|               404|           28 St (6)|  Manhattan| 40.74307| -73.98426|         107|   Manhattan|          Gramercy|
|               257|    Sutphin Blvd (F)|     Queens| 40.70546| -73.81071|         130|      Queens|           Jamaica|
|               367|  St Lawrence Av (6)|      Bronx| 40.83151|-73.867615|         212|       Bronx|Soundview/Bruckner|
|               151|    145 St (A,C,B,D)|  Manhattan|40.824783|-73.944214|         116|   Manhattan|  Hamilton Heights|
+------------------+--------------------

(424, 8)

In [ ]:
# Join PULocationID onto ridership
sdf_ridership_station = sdf_ridership.join(
    sdf_station_zones.select("station_complex_id", "PULocationID"),
    on="station_complex_id", # join using station_complex_id
    how="left"
)
sdf_ridership_station.show(5)
spark_shape(sdf_ridership_station)

+------------------+-------------------+--------------------+-------+---------+---------+---------+------------+
|station_complex_id|  transit_timestamp|     station_complex|borough|ridership| latitude|longitude|PULocationID|
+------------------+-------------------+--------------------+-------+---------+---------+---------+------------+
|                 1|2025-09-01 00:00:00|Astoria-Ditmars B...| Queens|     52.0|40.775036|-73.91203|         223|
|                 1|2026-03-01 00:00:00|Astoria-Ditmars B...| Queens|     62.0|40.775036|-73.91203|         223|
|                 1|2025-12-01 00:00:00|Astoria-Ditmars B...| Queens|     40.0|40.775036|-73.91203|         223|
|                 1|2026-01-01 00:00:00|Astoria-Ditmars B...| Queens|    134.0|40.775036|-73.91203|         223|
|                 1|2025-04-01 00:00:00|Astoria-Ditmars B...| Queens|     40.0|40.775036|-73.91203|         223|
+------------------+-------------------+--------------------+-------+---------+---------+-------

(5140101, 8)

## Extract MTA Ridership features
This section will join two key aggregated features from `mta_ridership` to the features df `sdf_tlc_hourly`: 
1. `n_stations`: the number of stations present in the `PULocationID` (taxi zone)
2. `ridership`: the number of total ridership for all stations in `PULocationID` (taxi zone) during the given `hour`

As useful columns for downstream readability, the following are also joined:
- `borough`: the name of the borough for which the `PULocationID` (taxi zone) belongs in, joined from `mta_ridership` dataset
- `taxi_zone`: the name of the taxi zone, joined from `taxi_zones` dataset

### 1. Join `n_stations`
 

In [ ]:
# Zone-level station features (from all zones)
sdf_zone_features = spark.createDataFrame(
    zones[["LocationID", "borough", "zone"]]
    .rename(columns={
        "LocationID": "PULocationID", # in accordance with taxi dataset
        "borough": "borough",
        "zone": "taxi_zone" # specify for clarity
    })
)

# Add number of subway station complexes per taxi zone to zone features
sdf_zone_features = (
    sdf_zone_features
    .join(
        sdf_station_zones
        .groupBy("PULocationID")
        .agg(F.countDistinct("station_complex_id").alias("n_stations")),
        on="PULocationID",
        how="left"
    )
    .fillna({"n_stations": 0}) # NULL if no record in sdf_zone_features table, i.e. no station in given zone
)

# Results in PULocationID, borough, taxi_zone, n_stations
sdf_zone_features.show(5)
spark_shape(sdf_zone_features)

+------------+-------------+--------------------+----------+
|PULocationID|      borough|           taxi_zone|n_stations|
+------------+-------------+--------------------+----------+
|           5|Staten Island|       Arden Heights|         0|
|           4|    Manhattan|       Alphabet City|         0|
|           6|Staten Island|Arrochar/Fort Wad...|         0|
|           1|          EWR|      Newark Airport|         0|
|           3|        Bronx|Allerton/Pelham G...|         2|
+------------+-------------+--------------------+----------+
only showing top 5 rows


(263, 4)

In [ ]:
# Add zone features to TLC hourly data
sdf_tlc_hourly = sdf_tlc_hourly.join(
    sdf_zone_features, # add any matching columns from sdf_zone_features
    on="PULocationID", # join on PULocationID
    how="left" # keep all rows from sdf_tlc_hourly
)

# Remove invalid zones 264 and 265 
sdf_tlc_hourly = sdf_tlc_hourly.filter(
    ~F.col("PULocationID").isin(264, 265)
)

# Results in an addition of 3 columns: borough, taxi_zone, n_stations
sdf_tlc_hourly.show(5)
spark_shape(sdf_tlc_hourly)

+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+
|PULocationID|               hour|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|      borough|           taxi_zone|n_stations|
+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+
|           4|2024-12-31 23:00:00|           0|           0.00|            1|           30.17|              130.0|    Manhattan|       Alphabet City|         0|
|           6|2024-12-31 23:00:00|           0|           0.00|            1|            23.1|              111.0|Staten Island|Arrochar/Fort Wad...|         0|
|           7|2024-12-31 23:00:00|           0|           0.00|            7|          267.57|             288.14|       Queens|             Astoria|         5|
|          10|2024-12-31 23:00:00|

(3139662, 10)

### 2. Join `ridership` by (PULocationID, hour)

In [ ]:
# Aggregate for hourly ridership in each zone
sdf_ridership_zone = (
    sdf_ridership_station # hourly ridership by each station (with zone column to join on)
    .withColumn("hour", F.date_trunc("hour", F.col("transit_timestamp")))
    .groupBy("hour", "PULocationID")
    .agg(F.sum("ridership").alias("ridership"))
)
sdf_ridership_zone.show(5)
spark_shape(sdf_ridership_zone)

+-------------------+------------+---------+
|               hour|PULocationID|ridership|
+-------------------+------------+---------+
|2025-09-12 09:00:00|         153|    217.0|
|2025-09-04 22:00:00|          36|   1041.0|
|2025-09-07 14:00:00|         129|   2131.0|
|2025-09-29 03:00:00|         129|    171.0|
|2025-09-02 00:00:00|         232|     57.0|
+-------------------+------------+---------+
only showing top 5 rows


(1861510, 3)

In [ ]:
# Add hourly ridership in each zone to TLC hourly 
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

sdf_tlc_hourly = sdf_tlc_hourly.join(
    sdf_ridership_zone,
    on=["hour", "PULocationID"],
    how="left"
).fillna({"ridership": 0})

sdf_tlc_hourly.show(5) 
spark_shape(sdf_tlc_hourly)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+---------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|      borough|           taxi_zone|n_stations|ridership|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+---------+
|2024-12-31 23:00:00|           4|           0|           0.00|            1|           30.17|              130.0|    Manhattan|       Alphabet City|         0|      0.0|
|2024-12-31 23:00:00|           6|           0|           0.00|            1|            23.1|              111.0|Staten Island|Arrochar/Fort Wad...|         0|      0.0|
|2024-12-31 23:00:00|          11|           0|           0.00|            2|           22.98|              347.5|     Brooklyn|          Bath Be

(3139662, 11)

In [40]:
# Save a checkpoint of the collated data, containing features from tlc_trips, mta_ridership
tlc_zone_hourly_v1_path = "../data/features/tlc_zone_hourly_v1"

(
    sdf_tlc_hourly
    .write
    .mode("overwrite")
    .option("parquet.enable.dictionary", "false") # more stable for local
    .parquet(tlc_zone_hourly_v1_path)
)

### Save above feature sdfs as parquets
Considering rerun time, this is a good checkpoint to save and reload while working on the next section.

In [41]:
# Save sdfs to parquet
save_sdf(sdf_taxi_hourly, "taxi_hourly")
save_sdf(sdf_fhvhv_hourly, "fhvhv_hourly")
save_sdf(sdf_station_zones, "station_zones")
save_sdf(sdf_ridership_station, "ridership_station")
save_sdf(sdf_zone_features, "zone_features")
save_sdf(sdf_ridership_zone, "ridership_zone")

Saved taxi_hourly to ../data/features/taxi_hourly


Saved fhvhv_hourly to ../data/features/fhvhv_hourly
Saved station_zones to ../data/features/station_zones


Saved ridership_station to ../data/features/ridership_station
Saved zone_features to ../data/features/zone_features


Saved ridership_zone to ../data/features/ridership_zone


In [42]:
# Read them (if needed)
sdf_taxi_hourly = read_sdf("taxi_hourly")
sdf_fhvhv_hourly = read_sdf("fhvhv_hourly")
sdf_tlc_hourly = read_sdf("tlc_zone_hourly_v1")
sdf_station_zones = read_sdf("station_zones")
sdf_ridership_station = read_sdf("ridership_station")
sdf_zone_features = read_sdf("zone_features")
sdf_ridership_zone = read_sdf("ridership_zone")

## Extract MTA Alerts features
This section is engineers and joins the following features:
- `disruption_severity`: an extracted feature from (3) which describes the severity of disruption in the given `PULocationID` during the given `hour`
- `n_alerts`: total number of alerts affecting any station in `PULocationID` , during the given `hour` 

To do so, we need a few intermediate sdfs: 
- `sdf_alerts_which_station`: extract station(s) (and hence zone(s)) affected by alert from `Header` and `Affected`
    - use diagnostic flags (Boolean): duplicate_station_name (e.g. 7 Av (F, G), 7 Av (B, Q)), no_station_match (no 1-1 match)
    - use a mapping from pipe-delimited status_label to discrete labelled severity 
- `sdf_alerts_zone_hour_events`: groupBy (PULocationID, hour, event_id), aggregate by max(disruption_severity), stations, zones
- `sdf_alerts_zone_hour_impact`: groupBy (PULocationID, hour), aggregate by n_alerts (unique events), sum(max_disruption_severity)
    - max_disruption_severity from each unique event is summed for the total disruption score in the zone

In [43]:
# Read MTA ridership
sdf_alerts = spark.read.parquet(alerts_path)
sdf_alerts.show(5)
spark_shape(sdf_alerts)

+--------+-------------+-------------------+--------------------+---------+--------------------+--------------------+
|event_id|update_number|               date|        status_label| affected|              header|         description|
+--------+-------------+-------------------+--------------------+---------+--------------------+--------------------+
|  202930|            0|2025-06-01 00:33:00|part-suspended | ...|    N | Q|There is limited ...|What's Happening?...|
|  202933|            0|2025-06-01 00:33:00|              delays|    F | R|Northbound F/R tr...|                NULL|
|  202930|            1|2025-06-01 00:38:00|part-suspended | ...|    N | Q|There is limited ...|What's Happening?...|
|  202933|            1|2025-06-01 00:49:00|              delays|    F | R|Northbound F/R tr...|                NULL|
|  202930|            2|2025-06-01 00:50:00|part-suspended | ...|N | Q | R|There is limited ...|What's Happening?...|
+--------+-------------+-------------------+------------

(67502, 7)

In [44]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Build station names
sdf_station_ref = (
    sdf_station_zones
    .select("station_complex_id", "station_complex", "PULocationID")
    .distinct()
    .withColumn("station_name", F.trim(F.regexp_extract("station_complex", r"^(.*)\s\(", 1)))
    .withColumn("station_routes", F.split(F.regexp_extract("station_complex", r"\((.*)\)$", 1), ",\\s*"))
)
sdf_station_ref.show(5, truncate=False)
spark_shape(sdf_station_ref)

+------------------+------------------------------------------+------------+----------------------------------+--------------+
|station_complex_id|station_complex                           |PULocationID|station_name                      |station_routes|
+------------------+------------------------------------------+------------+----------------------------------+--------------+
|294               |238 St (1)                                |220         |238 St                            |[1]           |
|372               |Longwood Av (6)                           |147         |Longwood Av                       |[6]           |
|235               |York St (F)                               |66          |York St                           |[F]           |
|28                |Union St (R)                              |106         |Union St                          |[R]           |
|279               |Sutphin Blvd-Archer Av-JFK Airport (E,J,Z)|130         |Sutphin Blvd-Archer Av-JFK Airport|

(424, 5)

In [45]:
# Alerts reference: time truncated to hourly, affected_severity listed, disruption_severity mapped
sdf_alerts_base = (
    sdf_alerts
    .withColumn("hour", F.date_trunc("hour", F.col("date")))
    .withColumn("affected_routes", F.split(F.regexp_replace("affected", " ", ""), "\\|")) # array version for matching later
    .withColumn("status_labels", F.split(F.regexp_replace("status_label", " ", ""), "\\|"))
    .withColumn(
        "disruption_severity",
        F.greatest(
            F.when(F.array_contains("status_labels", "delays"), 1).otherwise(0),
            F.when(F.array_contains("status_labels", "boarding-change"), 1).otherwise(0),
            F.when(F.array_contains("status_labels", "special-notice"), 1).otherwise(0),
            F.when(F.array_contains("status_labels", "stops-skipped"), 2).otherwise(0),
            F.when(F.array_contains("status_labels", "reroute"), 2).otherwise(0),
            F.when(F.array_contains("status_labels", "express-to-local"), 2).otherwise(0),
            F.when(F.array_contains("status_labels", "severe-delays"), 3).otherwise(0),
            F.when(F.array_contains("status_labels", "part-suspended"), 3).otherwise(0),
            F.when(F.array_contains("status_labels", "suspended"), 4).otherwise(0),
            F.when(F.array_contains("status_labels", "cancellations"), 4).otherwise(0),
        )
    )
)
sdf_alerts_base.show(truncate=False)
spark_shape(sdf_alerts_base)

+--------+-------------+-------------------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------

(67502, 11)

In [46]:
# Cross join alerts to station names, filter for where station name is mentioned anywhere in `header`
sdf_alert_station_candidates = (
    sdf_alerts_base
    .select("event_id", "update_number", "hour", "date", "status_label", "affected",
            "affected_routes", "header", "disruption_severity")
    .crossJoin(F.broadcast(sdf_station_ref)) # cross join by broadcastingsmall table to avoid big shuffle
    .withColumn("match_start", F.instr(F.lower("header"), F.lower("station_name")))
    .filter(F.col("match_start") > 0)
    .withColumn("match_end", F.col("match_start") + F.length("station_name") - 1)
)

In [47]:
# Identify shorter matches within longer matches for the same alert update
a = sdf_alert_station_candidates.alias("a")
b = sdf_alert_station_candidates.alias("b")

# Identify matches which are duplicated for shorter matches within longer matches
sdf_bad_matches = (
    a.join(
        b,
        (F.col("a.event_id") == F.col("b.event_id")) &
        (F.col("a.update_number") == F.col("b.update_number")) &
        (F.length("a.station_name") < F.length("b.station_name")) &
        (F.col("a.match_start") >= F.col("b.match_start")) &
        (F.col("a.match_end") <= F.col("b.match_end")),
        "inner"
    )
    .select(
        F.col("a.event_id").alias("event_id"),
        F.col("a.update_number").alias("update_number"),
        F.col("a.station_complex_id").alias("station_complex_id"),
    )
    .distinct()
)

# Remove bad substring matches
sdf_alert_station_candidates = (
    sdf_alert_station_candidates
    .join(
        sdf_bad_matches,
        on=["event_id", "update_number", "station_complex_id"],
        how="left_anti"
    )
    .drop("match_start", "match_end")
)

sdf_alert_station_candidates.show(truncate=False)
spark_shape(sdf_alert_station_candidates)

+--------+-------------+------------------+-------------------+-------------------+----------------+---------+---------------+--------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+---------------------------------+------------+----------------------+---------------------+
|event_id|update_number|station_complex_id|hour               |date               |status_label    |affected |affected_routes|header                                                                                                                                            |disruption_severity|station_complex                  |PULocationID|station_name          |station_routes       |
+--------+-------------+------------------+-------------------+-------------------+----------------+---------+---------------+------------------------------------------------------------------------------------------------------

(123983, 14)

In [48]:
# If only one station matched, keep match; If more than one station matched, keep match iff route matches
w_station_name = Window.partitionBy("station_name")

# Which station names refer to multiple station complexes?
sdf_duplicate_station_names = (
    sdf_station_ref
    .groupBy("station_name")
    .agg(
        (F.countDistinct("station_complex_id") > 1).alias("duplicate_station_name")
    )
)


sdf_alerts_which_station = (
    sdf_alert_station_candidates
    .join(
        F.broadcast(sdf_duplicate_station_names),
        on="station_name",
        how="left"
    )
    .withColumn(
        "route_match",
        F.size(F.array_intersect("affected_routes", "station_routes")) > 0
    )
    .filter(
        (~F.col("duplicate_station_name")) | F.col("route_match") # keep if station name not duplicated OR there's a route match
    )
)

sdf_alerts_which_station.show(truncate=False)
spark_shape(sdf_alerts_which_station)

+------------------------------+--------+-------------+------------------+-------------------+-------------------+----------------+--------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------+-------------------+-----------------------------------+------------+------------------------+----------------------+-----------+
|station_name                  |event_id|update_number|station_complex_id|hour               |date               |status_label    |affected|affected_routes|header                                                                                                                                         |disruption_severity|station_complex                    |PULocationID|station_routes          |duplicate_station_name|route_match|
+------------------------------+--------+-------------+------------------+-------------------+-------------------+----------------+--------+

(81145, 16)

In [49]:
save_sdf(sdf_alerts_which_station, "alerts_which_station")
sdf_alerts_which_station = read_sdf("alerts_which_station")

Saved alerts_which_station to ../data/features/alerts_which_station


In [ ]:
sdf_alerts_which_station.filter(F.col("disruption_severity") == 4).show(truncate=False)

+------------------------+--------+-------------+------------------+-------------------+-------------------+-------------------------------------+-------------+---------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+---------------------------------+------------+------------------------+----------------------+-----------+
|station_name            |event_id|update_number|station_complex_id|hour               |date               |status_label                         |affected     |affected_routes|header                                                                                                                                                         |disruption_severity|station_complex                  |PULocationID|station_routes          |duplicate_station_name|route_match|
+------------------------+--------+-------------+------------------+----

In [50]:
# Aggregate to (PULocationID, hour, event)
sdf_alerts_zone_hour_events = (
    sdf_alerts_which_station
    .groupBy("PULocationID", "hour", "event_id")
    .agg(
        F.max("disruption_severity").alias("max_disruption_severity"),
        F.collect_set("station_complex").alias("matched_stations"),
        F.collect_set("station_complex_id").alias("matched_station_ids"),
    )
)
sdf_alerts_zone_hour_events.show(truncate=False)
spark_shape(sdf_alerts_zone_hour_events)

+------------+-------------------+--------+-----------------------+-----------------+-------------------+
|PULocationID|hour               |event_id|max_disruption_severity|matched_stations |matched_station_ids|
+------------+-------------------+--------+-----------------------+-----------------+-------------------+
|3           |2025-01-04 04:00:00|185353  |1                      |[Gun Hill Rd (5)]|[444]              |
|3           |2025-01-07 15:00:00|185662  |1                      |[Pelham Pkwy (5)]|[445]              |
|3           |2025-01-21 00:00:00|187137  |2                      |[Gun Hill Rd (5)]|[444]              |
|3           |2025-01-31 10:00:00|188422  |1                      |[Gun Hill Rd (5)]|[444]              |
|3           |2025-02-12 03:00:00|189876  |1                      |[Pelham Pkwy (5)]|[445]              |
|3           |2025-02-12 19:00:00|189973  |1                      |[Pelham Pkwy (5)]|[445]              |
|3           |2025-02-13 02:00:00|190011  |1  

(52170, 6)

In [51]:
sdf_alerts_zone_hour_impact = (
    sdf_alerts_zone_hour_events
    .groupBy("PULocationID", "hour")
    .agg(
        F.countDistinct("event_id").alias("n_alerts"),
        F.sum("max_disruption_severity").alias("disruption_severity"),
    )
)
sdf_alerts_zone_hour_impact.show(truncate=False)
spark_shape(sdf_alerts_zone_hour_impact)

+------------+-------------------+--------+-------------------+
|PULocationID|hour               |n_alerts|disruption_severity|
+------------+-------------------+--------+-------------------+
|238         |2026-03-11 12:00:00|1       |1                  |
|230         |2026-04-06 05:00:00|1       |2                  |
|26          |2026-02-19 07:00:00|1       |2                  |
|212         |2026-02-27 13:00:00|1       |2                  |
|65          |2025-11-29 12:00:00|1       |1                  |
|25          |2025-01-23 02:00:00|1       |1                  |
|129         |2025-03-02 01:00:00|1       |1                  |
|234         |2026-02-16 18:00:00|1       |2                  |
|144         |2025-09-14 02:00:00|1       |2                  |
|236         |2025-06-19 23:00:00|1       |1                  |
|78          |2026-01-29 12:00:00|1       |2                  |
|162         |2026-01-13 00:00:00|1       |2                  |
|74          |2025-11-03 14:00:00|1     

(50175, 4)

In [52]:
sdf_alerts_zone_hour_impact.filter(F.col("n_alerts") > 3).show

<bound method DataFrame.show of +------------+-------------------+--------+-------------------+
|PULocationID|               hour|n_alerts|disruption_severity|
+------------+-------------------+--------+-------------------+
|         181|2026-02-03 12:00:00|       4|                  6|
|         249|2026-02-05 22:00:00|       5|                  8|
|          65|2025-06-26 23:00:00|       4|                  7|
|         161|2025-02-14 21:00:00|       4|                  4|
|         249|2026-01-12 17:00:00|       4|                  7|
|          66|2026-03-28 11:00:00|       4|                  4|
|         186|2025-09-25 17:00:00|       4|                  9|
|         249|2025-07-29 22:00:00|       4|                  4|
|         249|2025-07-29 17:00:00|       4|                  6|
|         168|2025-03-19 12:00:00|       4|                  9|
|          97|2026-04-20 12:00:00|       4|                  8|
|          97|2026-04-20 13:00:00|       5|                  9|
|       

In [53]:
# Join alerts features (n_alerts, disruption_severity) per zone, hour 
sdf_tlc_hourly = (
    sdf_tlc_hourly
    .drop("n_alerts", "disruption_severity") # reruns safe
    .join(
        sdf_alerts_zone_hour_impact,
        on=["PULocationID", "hour"],
        how="left"
    )
    .fillna({
        "n_alerts": 0,
        "disruption_severity": 0,
    })
)

sdf_tlc_hourly.show(truncate=False)
spark_shape(sdf_tlc_hourly)

+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+-----------------------+----------+---------+--------+-------------------+
|PULocationID|hour               |n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|borough      |taxi_zone              |n_stations|ridership|n_alerts|disruption_severity|
+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+-----------------------+----------+---------+--------+-------------------+
|3           |2025-01-01 00:00:00|0           |0.00           |90           |2700.74         |456.62             |Bronx        |Allerton/Pelham Gardens|2         |12       |0       |0                  |
|4           |2025-01-01 00:00:00|15          |311.80         |277          |10695.69        |400.6              |Manhattan    |Alphabet City          |0         |0        |0       |0     

(3138172, 13)

In [ ]:
save_sdf(sdf_tlc_hourly, "tlc_zone_hourly_v2")
sdf_tlc_hourly = read_sdf("tlc_zone_hourly_v2")

## Extract Meteostat Weather features
This section will join one feature from `meteostat_weather` to the features df `sdf_tlc_hourly`: 
- `weather_severity`: the weather condition code `coco` from dataset mapped to a severity scale on 0-3

In [7]:
# Read Meteostat Weather data
sdf_weather = spark.read.parquet(weather_path)

In [8]:
sdf_weather_hourly = (
    sdf_weather
    .withColumn("hour", F.date_trunc("hour", F.col("time")))
    .select("hour", "coco")
    .withColumn(
        "weather_severity",
        F.when(F.col("coco").isin(1, 2, 3, 4), 0)                   # clear / partly cloudy / cloudy / overcast
         .when(F.col("coco").isin(5, 7, 8, 18, 19, 20), 1)          # drizzle / rain / rain showers / mist / smoke/haze / dust/sand
         .when(F.col("coco").isin(9, 11, 12, 17), 2)                # snow / snow showers / fog
         .when(F.col("coco").isin(6, 9, 10, 13, 14, 15, 16), 3)     # freezing drizzle / thunderstorms / freezing rain / blowing snow / snow grains / ice pellets / hail
         .otherwise(0)
    )
)

In [9]:
# Join the hourly weather data
sdf_tlc_hourly = (
    sdf_tlc_hourly
    .drop("coco", "weather_severity")
    .join(sdf_weather_hourly, on="hour", how="left")
    .fillna({"weather_severity": 0})
    .orderBy("hour", "PULocationID")
)

sdf_tlc_hourly.show(truncate=False)
spark_shape(sdf_tlc_hourly)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+-----------------------+----------+---------+--------+-------------------+----+----------------+
|hour               |PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|borough      |taxi_zone              |n_stations|ridership|n_alerts|disruption_severity|coco|weather_severity|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+-----------------------+----------+---------+--------+-------------------+----+----------------+
|2025-01-01 00:00:00|3           |0           |0.00           |90           |2700.74         |456.62             |Bronx        |Allerton/Pelham Gardens|2         |12       |0       |0                  |8   |1               |
|2025-01-01 00:00:00|4           |15          |311.80         |277          |10695.69        |400.6 

(3138172, 15)

In [10]:
sdf_tlc_hourly.filter(F.col("disruption_severity")>3).show(50, truncate=False)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+---------+-----------------------------+----------+---------+--------+-------------------+----+----------------+
|hour               |PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|borough  |taxi_zone                    |n_stations|ridership|n_alerts|disruption_severity|coco|weather_severity|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+---------+-----------------------------+----------+---------+--------+-------------------+----+----------------+
|2025-01-03 07:00:00|186         |115         |1763.34        |77           |2961.13         |156.06             |Manhattan|Penn Station/Madison Sq West |3         |5948     |2       |4                  |1   |0               |
|2025-01-03 08:00:00|133         |0           |0.00           |95           |2326.86        

In [ ]:
save_sdf(sdf_tlc_hourly, "tlc_zone_hourly_v3")
sdf_tlc_hourly = read_sdf("tlc_zone_hourly_v3")

[Stage 14:>                                                         (0 + 8) / 8]
Saved tlc_zone_hourly_v3 to ../data/features/tlc_zone_hourly_v3



## Final compilation for `sdf_tlc_hourly`
Only retain rows (PULocationID, hour) containing at least one MTA subway station. 


In [ ]:
# Filter out any zones without at least one MTA Subway station, then save
sdf_tlc_hourly_final = sdf_tlc_hourly.filter(F.col("n_stations") > 0)
save_sdf(sdf_tlc_hourly_final, "tlc_zone_hourly_final")
sdf_tlc_hourly_final = read_sdf("tlc_zone_hourly_final")

In [ ]:
# How many removed?
sdf_tlc_hourly.count()
sdf_tlc_hourly_final.count()

3117327
1239781


## Checkpoint
By the end of this Jupyter Notebook, all the relevant features should have been extracted from preprocessed data sets or engineered from a combination of features. Intermediate and final sdfs are saved as parquet directories using PySpark.

The main features parquet that will be used for analysis and modelling will be `tlc_zone_hourly_v3`, containing the key features: 
- `hour`: time at hourly grain, between 2025 January and 2026 May (inclusive)
- `PULocationID`: taxi zone (of pick up)
- `n_trips_taxi` and `gross_fare_taxi`
- `n_trips_fhvhv` and `gross_fare_fhvhv` and `avg_wait_time_fhvhv`
- `n_stations`: number of MTA stations in given zone 
- `ridership`: number of MTA subway riders in given zone, hour
- `n_alerts`: number of alerts (unique events) in given zone, hour
- `disruption_severity`: sum of maximum disruption severity (mapped) of unique events in given zone, hour
- `weather_severity`: mapped weather condition code

And supporting features for analysis and interpretation
- `borough`: borough name
- `taxi_zone`: taxi zone name
- `coco`: weather condition code 
